# 관계형 테이블과 키

고객과 주문을 키로 연결하고 조인 행 수가 늘어나는 원인을 확인합니다.

## 실행 환경 준비

In [1]:
# 표 조인과 키 검사를 위해 pandas를 불러옵니다.
import pandas as pd
from pandas.errors import MergeError

print("pandas", pd.__version__)

pandas 3.0.5


## 기본 실행

고객 표의 `customer_id`는 한 고객을 유일하게 구분하는 기본 키입니다. 주문 표의 같은 열은 고객 표를 가리키는 외래 키이며 한 고객에게 여러 주문이 있으므로 반복될 수 있습니다. 두 DataFrame을 입력으로 준비합니다.

In [2]:
# 고객의 기본 키와 주문의 외래 키 역할을 분리해 표를 준비합니다.
customers = pd.DataFrame([
 {"customer_id":"C001", "name":"김하나"},
 {"customer_id":"C002", "name":"이두리"},
])
orders = pd.DataFrame([
 {"order_id":"O101", "customer_id":"C001", "amount":32000},
 {"order_id":"O102", "customer_id":"C002", "amount":18000},
 {"order_id":"O103", "customer_id":"C001", "amount":27000},
])
print(customers.to_string(index=False))
print(orders.to_string(index=False))

customer_id name
       C001  김하나
       C002  이두리
order_id customer_id  amount
    O101        C001   32000
    O102        C002   18000
    O103        C001   27000


## 중간 결과 관찰

`orders.merge(customers, on="customer_id")`는 양쪽 키가 같은 행의 열을 결합합니다. 기본 내부 조인은 양쪽에 모두 있는 키만 남깁니다. `validate="many_to_one"`은 왼쪽 주문 키의 반복은 허용하고 오른쪽 고객 키는 유일한지 검사합니다. 반환된 DataFrame의 행 수와 고객 이름을 확인합니다.

In [3]:
# many_to_one은 오른쪽 고객 키가 유일한지 검사합니다.
joined = orders.merge(customers, on="customer_id", validate="many_to_one")
print(joined[["order_id","name","amount"]].to_string(index=False))
assert len(joined) == len(orders)

order_id name  amount
    O101  김하나   32000
    O102  이두리   18000
    O103  김하나   27000


## 값 하나를 바꿔 비교

고객 표에 없는 C999 주문을 추가하고 `how="left"`로 왼쪽 조인을 실행합니다. 왼쪽 주문은 유지되고 찾지 못한 고객 열에는 결측값이 생깁니다. `indicator=True`가 만든 `_merge` 열에서 `left_only`를 선택해 연결되지 않은 외래 키를 찾습니다.

In [4]:
# 부모 고객이 없는 주문은 왼쪽 조인의 표시 열로 발견합니다.
with_orphan = pd.concat([orders, pd.DataFrame([
    {"order_id":"O104", "customer_id":"C999", "amount":9000}
])])
checked = with_orphan.merge(customers, on="customer_id", how="left", indicator=True)
print(checked.loc[checked["_merge"] == "left_only",
                  ["order_id", "customer_id", "name", "_merge"]].to_string(index=False))
assert checked.loc[checked["_merge"] == "left_only", "order_id"].tolist() == ["O104"]

order_id customer_id name    _merge
    O104        C999  NaN left_only


## 실패 재현

고객 키가 중복되면 주문 한 행이 두 행으로 복제됩니다.

In [5]:
# 중복 고객 키는 주문을 복제하며 관계 검증에서도 거부됩니다.
duplicated = pd.concat([customers, pd.DataFrame([
    {"customer_id":"C001", "name":"김하나(중복)"}
])], ignore_index=True)
wrong = orders.merge(duplicated, on="customer_id")
print("주문 행:", len(orders), "조인 행:", len(wrong))
try:
    orders.merge(duplicated, on="customer_id", validate="many_to_one")
except MergeError as error:
    print(type(error).__name__, "중복 키 거부")
else:
    raise AssertionError("중복 키가 거부되지 않았습니다.")
assert len(wrong) == 5

주문 행: 3 조인 행: 5
MergeError 중복 키 거부


## 수정

조인 전에 키의 유일성을 검사하고 many_to_one 규칙을 선언합니다.

In [6]:
# 교육 데이터는 먼저 온 원본을 유지한다는 규칙으로 중복을 정리합니다.
deduplicated = duplicated.drop_duplicates("customer_id", keep="first")
assert deduplicated["customer_id"].is_unique
fixed = orders.merge(deduplicated, on="customer_id", validate="many_to_one")
print("수정 후 조인 행:", len(fixed), "C001 이름:", fixed.iloc[0]["name"])
assert len(fixed) == 3 and fixed.iloc[0]["name"] == "김하나"

수정 후 조인 행: 3 C001 이름: 김하나
